"""
automate_Muhammad_Rahman.py
============================
Automated preprocessing pipeline for image classification
Dataset: Sampah Daur Ulang (Kaggle)

Konversi dari proses eksperimen pada notebook Eksperimen_SML_Muhammad-Rahman.py
menjadi fungsi-fungsi otomatis yang mengembalikan data siap dilatih.

Tahapan:
  1. Download dataset dari Kaggle
  2. Ekstraksi file ZIP
  3. Penemuan struktur kelas secara otomatis
  4. Split dataset menjadi Train / Validation / Test (70/15/15)
  5. Membangun tf.data.Dataset pipeline siap latih

Usage:
    from automate_Muhammad_Rahman import preprocess
    train_ds, val_ds, test_ds, class_names, config = preprocess()
"""

In [1]:
import os
import sys
import shutil
import random
import zipfile
import subprocess
from pathlib import Path

import numpy as np
import tensorflow as tf

# ======================== CONFIGURATION ========================
SEED = 42
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
TRAIN_SPLIT = 0.70
VAL_SPLIT = 0.15
TEST_SPLIT = 0.15
DATASET_SLUG = "fathurrahmanalfarizy/sampah-daur-ulang"
ZIP_NAME = "sampah-daur-ulang.zip"
EXTRACT_DIR_NAME = "sampah-daur-ulang"
IMG_EXTS = {".jpg", ".jpeg", ".png"}

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)


ImportError: cannot import name 'KerasLazyLoader' from 'tensorflow.python.util.lazy_loader' (/home/brayone-xv/anaconda3/envs/sml_system/lib/python3.12/site-packages/tensorflow/python/util/lazy_loader.py)

In [3]:
# ======================== DATA LOADING ========================

def setup_kaggle_credentials():
    """Mencari kaggle.json dan mengkonfigurasi Kaggle API."""
    kaggle_json_source = Path.home() / ".kaggle" / "kaggle.json"
    candidates = [
        Path("kaggle.json"),
        Path.cwd() / "kaggle.json",
        kaggle_json_source,
    ]
    found = None
    for candidate in candidates:
        if candidate.exists():
            found = candidate
            break
    if found is None:
        raise FileNotFoundError(
            "kaggle.json tidak ditemukan. "
            "Letakkan file di folder kerja atau di ~/.kaggle/kaggle.json."
        )
    kaggle_dir = Path.home() / ".kaggle"
    kaggle_dir.mkdir(parents=True, exist_ok=True)
    target = kaggle_dir / "kaggle.json"
    if found.resolve() != target.resolve():
        shutil.copy2(found, target)
    os.chmod(target, 0o600)
    os.environ["KAGGLE_CONFIG_DIR"] = str(kaggle_dir)
    print(f"Kaggle config: {target}")


def ensure_kaggle_installed():
    """Install kaggle CLI jika belum tersedia."""
    try:
        import kaggle  # noqa: F401
    except ImportError:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", "kaggle"]
        )


def download_dataset(output_dir="."):
    """Download dataset dari Kaggle."""
    setup_kaggle_credentials()
    ensure_kaggle_installed()
    kaggle_cli = Path(sys.executable).with_name("kaggle")
    if not kaggle_cli.exists():
        kaggle_cli = Path(shutil.which("kaggle") or "kaggle")
    subprocess.check_call([
        str(kaggle_cli), "datasets", "download",
        "-d", DATASET_SLUG, "-p", output_dir, "--force",
    ])
    zip_path = Path(output_dir) / ZIP_NAME
    if not zip_path.exists():
        raise FileNotFoundError(f"Download gagal: {zip_path} tidak ditemukan.")
    print(f"Dataset berhasil didownload: {zip_path}")
    return zip_path


def extract_dataset(zip_path, extract_dir=None):
    """Ekstraksi file ZIP dataset."""
    if extract_dir is None:
        extract_dir = Path(zip_path).parent / EXTRACT_DIR_NAME
    extract_dir = Path(extract_dir)
    extract_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(extract_dir)
    print(f"Ekstraksi selesai: {extract_dir}")
    return extract_dir.resolve()



In [4]:
# ======================== EDA HELPERS ========================

def has_images(path):
    """Cek apakah sebuah direktori mengandung file gambar."""
    return any(
        p.is_file() and p.suffix.lower() in IMG_EXTS for p in path.iterdir()
    )


def is_class_root(path):
    """Cek apakah direktori merupakan root kelas (≥2 subfolder dengan gambar)."""
    subdirs = [d for d in path.iterdir() if d.is_dir()]
    if len(subdirs) < 2:
        return False
    return sum(1 for d in subdirs if has_images(d)) >= 2


def find_class_root(root):
    """Temukan direktori yang berisi subfolder kelas."""
    for p in root.rglob("*"):
        if p.is_dir() and is_class_root(p):
            return p
    return None


def print_dataset_info(class_root, class_names, all_paths, all_labels):
    """Cetak informasi EDA dasar tentang dataset."""
    print(f"\nClass root: {class_root}")
    print(f"Jumlah kelas: {len(class_names)}")
    print(f"Nama kelas: {class_names}")
    print(f"Total gambar: {len(all_paths)}")
    print("\nDistribusi kelas:")
    for idx, name in enumerate(class_names):
        count = int(np.sum(all_labels == idx))
        print(f"  {name}: {count} gambar")
    print()



In [5]:
# ======================== PREPROCESSING ========================

def list_images(base_dir, class_names):
    """Kumpulkan semua path gambar dan label dari direktori kelas."""
    paths, labels = [], []
    for idx, class_name in enumerate(class_names):
        class_dir = Path(base_dir) / class_name
        if not class_dir.exists():
            continue
        for p in class_dir.rglob("*"):
            if p.is_file() and p.suffix.lower() in IMG_EXTS:
                paths.append(str(p))
                labels.append(idx)
    return np.array(paths), np.array(labels)


def split_data(paths, labels):
    """Split data menjadi train, validation, dan test."""
    idx = np.arange(len(paths))
    np.random.shuffle(idx)
    paths, labels = paths[idx], labels[idx]

    n = len(paths)
    n_train = int(n * TRAIN_SPLIT)
    n_val = int(n * VAL_SPLIT)

    splits = {
        "train": (paths[:n_train], labels[:n_train]),
        "val": (paths[n_train:n_train + n_val], labels[n_train:n_train + n_val]),
        "test": (paths[n_train + n_val:], labels[n_train + n_val:]),
    }
    for name, (p, _) in splits.items():
        print(f"  {name}: {len(p)} gambar")
    return splits


def decode_img(path, label):
    """Decode dan resize gambar."""
    img = tf.io.read_file(path)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, IMG_SIZE)
    img = tf.cast(img, tf.float32)
    return img, label


def build_dataset(paths, labels, shuffle=False, batch_size=BATCH_SIZE):
    """Bangun tf.data.Dataset dari paths dan labels."""
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if shuffle:
        ds = ds.shuffle(min(len(paths), 1000), seed=SEED)
    ds = ds.map(decode_img, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds


def make_data_augmentation():
    """Buat layer augmentasi data."""
    return tf.keras.Sequential([
        tf.keras.layers.RandomFlip("horizontal"),
        tf.keras.layers.RandomRotation(0.15),
        tf.keras.layers.RandomZoom(0.15),
        tf.keras.layers.RandomContrast(0.2),
    ], name="data_augmentation")




In [6]:
# ======================== MAIN FUNCTION ========================

def preprocess(data_dir=None, download_data=True):
    """
    Pipeline preprocessing lengkap.

    Parameters
    ----------
    data_dir : str or Path, optional
        Path ke dataset yang sudah diekstrak.
        Jika None dan download_data=True, dataset didownload dari Kaggle.
    download_data : bool
        Apakah perlu download dataset jika data_dir tidak diberikan.

    Returns
    -------
    train_ds : tf.data.Dataset   — Dataset training (sudah di-batch & prefetch)
    val_ds   : tf.data.Dataset   — Dataset validasi
    test_ds  : tf.data.Dataset   — Dataset testing
    class_names : list[str]      — Daftar nama kelas
    config   : dict              — Konfigurasi (IMG_SIZE, BATCH_SIZE, NUM_CLASSES)
    """
    # 1. Data Loading
    if data_dir is not None:
        dataset_root = Path(data_dir).resolve()
    elif download_data:
        zip_path = download_dataset()
        dataset_root = extract_dataset(zip_path)
    else:
        raise ValueError("Berikan data_dir atau set download_data=True.")

    # 2. Temukan class root
    class_root = find_class_root(dataset_root)
    if class_root is None:
        raise RuntimeError("Tidak dapat menemukan folder kelas. Periksa struktur dataset.")

    # 3. EDA — Discover classes
    class_names = sorted([d.name for d in class_root.iterdir() if d.is_dir()])
    num_classes = len(class_names)
    if num_classes == 0:
        raise RuntimeError("Tidak ada folder kelas ditemukan.")

    # 4. List & split images
    all_paths, all_labels = list_images(class_root, class_names)
    if len(all_paths) == 0:
        raise RuntimeError("Tidak ada gambar ditemukan.")

    print_dataset_info(class_root, class_names, all_paths, all_labels)

    print("Splitting data:")
    splits = split_data(all_paths, all_labels)

    # 5. Build tf.data pipelines
    train_ds = build_dataset(*splits["train"], shuffle=True)
    val_ds = build_dataset(*splits["val"], shuffle=False)
    test_ds = build_dataset(*splits["test"], shuffle=False)

    config = {
        "IMG_SIZE": IMG_SIZE,
        "BATCH_SIZE": BATCH_SIZE,
        "NUM_CLASSES": num_classes,
        "SEED": SEED,
    }

    print("\n✅ Preprocessing selesai! Data siap dilatih.")
    return train_ds, val_ds, test_ds, class_names, config


In [7]:
# ======================== ENTRY POINT ========================

if __name__ == "__main__":
    train_ds, val_ds, test_ds, class_names, cfg = preprocess()
    print(f"\nConfig: {cfg}")
    print(f"Classes: {class_names}")


Kaggle config: /home/brayone-xv/.kaggle/kaggle.json
Dataset URL: https://www.kaggle.com/datasets/fathurrahmanalfarizy/sampah-daur-ulang
License(s): unknown


100%|██████████| 98.5M/98.5M [01:26<00:00, 1.20MB/s]



Dataset berhasil didownload: sampah-daur-ulang.zip
Ekstraksi selesai: sampah-daur-ulang

Class root: /home/brayone-xv/Documents/My File/CODE/pijak-by-dicoding/SML_System_Muhammad Rahman/1-Preprocessing/sampah-daur-ulang/DATASETS
Jumlah kelas: 6
Nama kelas: ['Kaca', 'Kardus', 'Kertas', 'Logam', 'Plastik', 'Residu']
Total gambar: 7014

Distribusi kelas:
  Kaca: 1110 gambar
  Kardus: 624 gambar
  Kertas: 1807 gambar
  Logam: 1210 gambar
  Plastik: 1257 gambar
  Residu: 1006 gambar

Splitting data:
  train: 4909 gambar
  val: 1052 gambar
  test: 1053 gambar

✅ Preprocessing selesai! Data siap dilatih.

Config: {'IMG_SIZE': (224, 224), 'BATCH_SIZE': 32, 'NUM_CLASSES': 6, 'SEED': 42}
Classes: ['Kaca', 'Kardus', 'Kertas', 'Logam', 'Plastik', 'Residu']


E0000 00:00:1780670058.768715   71272 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)
